In [14]:
import os
import pandas as pd
import numpy as np
import torch
import librosa
from datasets import Dataset
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split

In [11]:
PATH_TO_FMA_SMALL = r"C:\Users\kinga\Desktop\studia\DS\ED_project\fma_small\fma_small"
PATH_TO_TRACKS_CSV = r"C:\Users\kinga\Desktop\studia\DS\ED_project\tracks.csv"
MODEL_ID = "adhityamw11/distilhubert-finetuned_distillhubert-ravdess"

In [9]:
paths = {}
for root, dirs, files in os.walk(PATH_TO_FMA_SMALL):
    for file in files:
        if file.endswith(".mp3"):
            try:
                track_id = int(file.split(".")[0])
                paths[track_id] = os.path.join(root, file)
            except: continue

print(f"Znaleziono {len(paths)} plików MP3.")

tracks = pd.read_csv(PATH_TO_TRACKS_CSV, index_col=0, header=[0, 1])
keep_tracks = tracks[tracks[('set', 'subset')] == 'small']
df = keep_tracks[('track', 'genre_top')].reset_index()
df.columns = ['track_id', 'genre']

df['path'] = df['track_id'].map(paths)
df = df.dropna(subset=['path']).reset_index(drop=True)

genres = sorted(df['genre'].unique().tolist())
label2id = {label: i for i, label in enumerate(genres)}
id2label = {i: label for i, label in enumerate(genres)}
df['label'] = df['genre'].map(label2id)

print(f"Załadowano {len(df)} utworów z {len(genres)} gatunków.")

Znaleziono 8000 plików MP3.
Załadowano 8000 utworów z 8 gatunków.


In [15]:
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)

model = AutoModelForAudioClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(genres),
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

Loading weights: 100%|██████████| 53/53 [00:00<00:00, 7570.69it/s]


In [16]:
raw_dataset = Dataset.from_pandas(df[['path', 'label']])

def preprocess_function(examples):
    audio_arrays = []
    for path in examples["path"]:
        try:
            y, _ = librosa.load(path, sr=16000, duration=10)
            audio_arrays.append(y)
        except:
            audio_arrays.append(np.zeros(16000 * 10))

    inputs = feature_extractor(
        audio_arrays, 
        sampling_rate=16000, 
        max_length=16000 * 10,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )
    return inputs

encoded_dataset = raw_dataset.map(
    preprocess_function, 
    batched=True, 
    batch_size=4, 
    remove_columns=["path"]
)

train_test = encoded_dataset.train_test_split(test_size=0.1)


Map:  55%|█████▌    | 4420/8000 [03:09<02:17, 25.95 examples/s]C:\Users\kinga\AppData\Local\Temp\ipykernel_7512\3169691770.py:7: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(path, sr=16000, duration=10)
c:\Users\kinga\AppData\Local\Programs\Python\Python313\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
Map: 100%|██████████| 8000/8000 [06:12<00:00, 21.47 examples/s]  


In [19]:
print("Uruchamianie treningu ...")

training_args = TrainingArguments(
    output_dir="./wyniki_modelu_muzycznego",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_test["train"],
    eval_dataset=train_test["test"],
    processing_class=feature_extractor,
)

# START UCZENIA
trainer.train()

# ZAPISANIE MODELU
model.save_pretrained("./finalny_model_muzyczny")
feature_extractor.save_pretrained("./finalny_model_muzyczny")
print("Trening zakończony pomyślnie. Model zapisany w folderze 'finalny_model_muzyczny'.")

Uruchamianie treningu ...


c:\Users\kinga\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 